In [6]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from sklearn.cluster import KMeans
from sklearn import svm
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.metrics import r2_score
from typing import Sequence
from dataclasses import dataclass
from typing import NamedTuple
from typing import Mapping
from typing import TypeAlias
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

sys.path.append('C:\\Users\\Mark\\PycharmProjects\\approximationMap')

from control.experiment_data import ExperimentData

In [14]:
FuelCombination: TypeAlias = str

class ApproxData(NamedTuple):
    fuel_consumption: np.ndarray
    additive_consumption: np.ndarray
    approximated_surface: np.ndarray


def _get_consts() -> tuple[pd.DataFrame, pd.DataFrame]:
    steam_params = pd.DataFrame(
        {
            "Cp": [2.139108184, 2.079572787],  # теплоемкость (Дж/(кг*К)) Cp
            "adiabat": [1.302022264, None],  # показатель адиабаты kg
            "mol_mass": [0.018, None],  # молярная масса (кг/моль)
            "G": [0.000222222, None],  # mass flowrate (kg/s) G
            "G_hr": [0.8, None],  # mass flowrate (кг/ч) G_hr
            "D_tube": [0.003, None],  # диаметр отверстия трубки
            "D_jet": [0.0006, None],  # диаметр отверстия форсунки
            "P": [7.4, 1.01],  # P (bar)
            "T": [523.15, 329.6083348],  # T (K)
            "t": [250, 56.60833484],  # t_C (C)
            "rho": [3.063916824, 0.663734924],  # rho (kg/m3)
            "U": [10.26072679, 931.3243951],  # U (m/s)
            "Q": [4.351728228, 20.08834076],  # Q (л/мин)
            "Y_jet": [0.002280162, 0.207]  # Yjet (N)
            },
        index=["input_tube", "jet"]
    )

    air_params = pd.DataFrame(
        {
            "Cp": [1.006, 1.006],  # теплоемкость (Дж/(кг*К)) Cp
            "adiabat": [1.4, None],  # показатель адиабаты kg
            "mol_mass": [0.029, None],  # молярная масса (кг/моль)
            "G": [0.000287778, None],  # mass flowrate (kg/s) G
            "G_hr": [1.036, None],  # mass flowrate (кг/ч) G_hr
            "D_tube": [0.003, None],  # диаметр отверстия трубки
            "D_jet": [0.0006, None],  # диаметр отверстия форсунки
            "P": [7.8, 1.01],  # P (bar)
            "T": [523.15, 291.7270737],  # T (K)
            "t": [250, 18.72707372],  # t_C (C)
            "rho": [5.20313803, 1.208207736],  # rho (kg/m3)
            "U": [7.824552641, 682.4105445],  # U (m/s)
            "Q": [3.318510208, 14.29114063],  # Q (л/мин)
            "Y_jet": [0.002251732, 0.196]  # Yjet (N)
            }, 
        index=["input_tube", "jet"]
    )
    return steam_params, air_params

def _replace_additive_column_with_J_column(dfs: Mapping[FuelCombination, ApproxData]) -> Mapping[FuelCombination, pd.DataFrame]:
    steam_params, air_params = _get_consts()
    
    for key in dfs:
        st_prm_tube = steam_params.loc['input_tube']
        st_prm_jet = steam_params.loc['jet']
    
        air_prm_tube = air_params.loc['input_tube']
        air_prm_jet = air_params.loc['jet']
    
        if 'steam' in key:
            F_fuel, F_steam, surf = dfs[key]
            
            P_abs = 7.8 * F_steam
            ro_tube = P_abs * 100_000 * st_prm_tube['mol_mass'] / st_prm_tube['T'] / 8.31
            ro_atm = ((st_prm_jet['P'] / P_abs) ** (1 / st_prm_tube['adiabat'])) * ro_tube
            T_atm = st_prm_jet['P'] * 100_000 / ro_atm / 8.31 * st_prm_tube['mol_mass']
            U_tube = F_steam / 3_600 / np.pi / ro_tube / st_prm_tube['D_tube'] / st_prm_tube['D_tube'] * 4
            U_atm = np.sqrt(-2 * st_prm_jet['Cp'] * 1_000 * T_atm + U_tube * U_tube + 2 * st_prm_tube['Cp'] * 1_000 * st_prm_tube['T'])
            J = U_atm * F_steam / 3_600
            dfs[key] = F_fuel, J, surf
            
        else:
            F_fuel, F_air, surf = dfs[key]
            
            P_abs = 7.8 * F_air
            ro_tube = P_abs * 100_000 * air_prm_tube['mol_mass'] / air_prm_tube['T'] / 8.31
            ro_atm = ((air_prm_jet['P'] / P_abs) ** (1 / air_prm_tube['adiabat'])) * ro_tube
            T_atm = air_prm_jet['P'] * 100_000 / ro_atm / 8.31 * air_prm_tube['mol_mass']
            U_tube = F_air / 3_600 / np.pi / ro_tube / air_prm_tube['D_tube'] / air_prm_tube['D_tube'] * 4
            U_atm = np.sqrt(-2 * air_prm_jet['Cp'] * 1_000 * T_atm + U_tube * U_tube + 2 * air_prm_tube['Cp'] * 1_000 * air_prm_tube['T'])
            J = U_atm * F_air / 3_600
            dfs[key] = F_fuel, J, surf
    return dfs

def get_rbf_data(J: bool=False) -> Mapping[FuelCombination, ApproxData]:
    CO_vars = get_CO_vars()
    rbf_data = {}
    
    for CO_var in CO_vars:
        var_name = '_'.join(CO_var)
        ex = ExperimentData()
        ex.get_experiment_data(*CO_var)
        rbf_data[var_name] = ex.get_rbf_data()
    return rbf_data  

class Variant(NamedTuple):
    fuel_name: str
    additive_name: str
    component_name: str

@dataclass
class Variants:
    variant: Variant

def get_CO_vars() -> Sequence[Variants]:
    all_vars = ExperimentData().get_all_available_variations()
    CO_vars = [var for var in all_vars if var[2] == 'CO']
    return CO_vars

class LinearCoeffs(NamedTuple):
    a: float
    b: float

class MinPoints(NamedTuple):
    x: np.ndarray
    y: np.ndarray
    
def _linear(x, a, b):
    return a * x + b


def _get_min_CO(data: Mapping[FuelCombination, ApproxData]) -> Mapping[FuelCombination, MinPoints]:
    min_CO = {}
    
    for fuel_combination in data:
        fuel, additive, surf = data[fuel_combination]
        df = pd.DataFrame(surf, columns=fuel, index=additive)
        df = df.idxmin(axis=1)
        min_x = list(df)
        min_y = list(df.index)
        min_CO[fuel_combination] = min_x, min_y
    return min_CO

def get_linear_coeffs_min_CO(data: Mapping[FuelCombination, ApproxData], J: bool=False) -> Mapping[FuelCombination, LinearCoeffs]:
    coeffs = {}
    if J:
        data = _replace_additive_column_with_J_column(data)
        min_CO = _get_min_CO(data)
    else:
        min_CO = _get_min_CO(data)
    
    for fuel_combination in min_CO:
        min_x, min_y = min_CO[fuel_combination]
        params, _ = curve_fit(_linear, min_x, min_y)
        a, b = params
        coeffs[fuel_combination] = a, b
        
    return coeffs

get_linear_coeffs_min_CO(get_rbf_data(), J=True)

{'diesel_air_CO': (0.3873509890345504, -0.23805342392892656),
 'diesel_steam_CO': (0.4027244444188116, -0.28430685264905775),
 'crude_oil_steam_CO': (0.6352922850719065, -0.3877065797809617),
 'heavy_oil_air_CO': (0.47092597526081886, -0.2114382464896674),
 'heavy_oil_steam_CO': (0.7592723434036563, -0.47478204594914364),
 'kerosene_air_CO': (0.5438224558915393, -0.5237698341635039),
 'kerosene_steam_CO': (0.9563097282127483, -1.038398783154443),
 'waste_oil_steam_CO': (0.3778633256572201, -0.24804260272129947)}

In [15]:
def show_result_equations() -> None:
    surfs = get_rbf_data()
    grad_coeffs = get_linear_coeffs_div_line(get_diff_data(), 4)
    min_coeffs = get_linear_coeffs_min_CO(surfs)
    mean_coeffs = get_mean_coeffs(grad_coeffs, min_coeffs)

    for fuel_combination in surfs:
        fuel_cons, add_cons_diff, diff = surfs[fuel_combination]
        grad_approx = _linear(fuel_cons, *grad_coeffs[fuel_combination])
        min_approx = _linear(fuel_cons, *min_coeffs[fuel_combination])
        mean_approx = _linear(fuel_cons, *mean_coeffs[fuel_combination])
        
        fig, ax = plt.subplots()
        plt.contourf(fuel_cons, add_cons_diff, diff)
        plt.plot(fuel_cons, grad_approx, c='black', linestyle='dotted', linewidth=2, label='нижняя линия макс. градиента')
        plt.plot(fuel_cons, min_approx, c='black', linestyle='dashed', linewidth=2, label='мин. концентрация CO')
        plt.plot(fuel_cons, mean_approx, c='orange', linestyle='dashdot', linewidth=2, label='среднее')
        clb = plt.colorbar()
        clb.ax.set_title('ppm')
        ax.set_xlabel('Расход топлива, кг/ч')
        ax.set_ylabel('Расход вводимого компонента, кг/ч')
        plt.legend()
        plt.ylim(min(add_cons_diff), max(add_cons_diff))
        plt.title(fuel_combination)
        plt.show()
        plt.close()
        
                                          
show_result_equations() 

NameError: name 'get_linear_coeffs_div_line' is not defined